# Cloud Provider Analytics — MVP (Segundo Parcial)
**Big Data 72.80 · ITBA · Arquitectura Lambda · PySpark + Structured Streaming + Cassandra/AstraDB**

Pipeline end-to-end mínimo: `Landing → Bronze → Silver → Gold → Serving (Cassandra)`, ejecutable de principio a fin en Google Colab.

## 0. Setup — AstraDB y credenciales
Cassandra es la capa de serving; AstraDB es Cassandra gestionado. La conexión usa un **Secure Connect Bundle (SCB)** + un **application token**.

1. En astra.datastax.com crear una base **Serverless (Non-Vector)** y un keyspace **`cloud_analytics`** desde la UI (en serverless no se puede `CREATE KEYSPACE` desde el driver; sí `CREATE TABLE`).
2. Generar un **application token** (rol *Database Administrator*) → copiar el string `AstraCS:...`.
3. Descargar el **Secure Connect Bundle** (zip) y subirlo a `/content/` (panel **Archivos → Subir**).
4. Subir el dataset para tener `/content/datalake/landing/` con los 7 CSV y `usage_events_stream/*.jsonl`.

### Credenciales por variables de entorno (no hardcodear)
Las credenciales se leen de **Colab Secrets** o de un archivo **`.env`** (ver `.env.example`). **Nunca** se escriben en el código ni se suben al repo (el `.gitignore` excluye `.env` y el `secure-connect-*.zip`). `/content` se borra entre sesiones; para persistir montá Drive.

In [ ]:
# --- 1. Instalación de dependencias ---
!pip -q install pyspark==3.5.1 cassandra-driver python-dotenv
print("instalado")

### 0b. (Opcional) Traer repo + datos automáticamente
Si se corre desde una sesión limpia de Colab, esta celda clona el repo y copia `datalake/landing/` a `/content/`, así no hay necesidad de subir los datos manualmente. Si ya se cargaron los datos, se puede saltear.

*Nota:* las credenciales de AstraDB (token + Secure Connect Bundle) **no** están en el repo (son secretos); esas se cargan aparte en Colab Secrets o `.env`.

In [ ]:
# Clona el repo y deja los datos en /content/datalake/landing/
import os, shutil
REPO_URL = "https://github.com/CamilaBelenLee/cloud_provider_analytics.git"
if not os.path.exists("/content/cloud_provider_analytics"):
    !git clone -q {REPO_URL} /content/cloud_provider_analytics
src = "/content/cloud_provider_analytics/datalake/landing"
dst = "/content/datalake/landing"
if os.path.exists(src) and not os.path.exists(dst):
    os.makedirs("/content/datalake", exist_ok=True)
    shutil.copytree(src, dst)
print("landing presente:", os.path.exists(dst),
      "| archivos:", len(os.listdir(dst)) if os.path.exists(dst) else 0)

In [ ]:
# --- 2. Sesión de Spark ---
# Sesión local: local[*] usa todos los núcleos de la VM de Colab. La misma API/plan
# corre igual en un cluster, así que el código no cambia si el dato crece.
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               LongType, IntegerType, BooleanType, DateType)
spark = (SparkSession.builder.appName("cloud_provider_analytics_mvp")
         .master("local[*]").config("spark.sql.shuffle.partitions","8").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
# --- 3. Rutas y zonas del Data Lake ---
import os
BASE="/content/datalake"; LANDING=f"{BASE}/landing"
BRONZE=f"{BASE}/bronze"; SILVER=f"{BASE}/silver"; GOLD=f"{BASE}/gold"
QUARANTINE=f"{BASE}/quarantine"; CHK="/content/checkpoints"
for p in [BRONZE,SILVER,GOLD,QUARANTINE,CHK]: os.makedirs(p, exist_ok=True)
EVENTS_DIR=f"{LANDING}/usage_events_stream"
print("landing:", os.path.exists(LANDING), "| events:", os.path.exists(EVENTS_DIR))

## 1. Esquemas explícitos
Usamos schema explícito (no `inferSchema`) porque con estos datos `inferSchema` rompe: `value` viene a veces como número y a veces como string `"100.0"`, y Spark lo leería como double y perdería los strings. El esquema de eventos es uno solo para v1 y v2 — `carbon_kg`/`genai_tokens` quedan nullable, así no hace falta código condicional. `value` se declara **string** y lo casteamos después. Ojo: el campo de tiempo se llama `timestamp`, no `event_time`.

In [ ]:
event_schema = StructType([
    StructField("event_id", StringType()), StructField("timestamp", StringType()),
    StructField("org_id", StringType()), StructField("resource_id", StringType()),
    StructField("service", StringType()), StructField("region", StringType()),
    StructField("metric", StringType()), StructField("value", StringType()),   # string -> cast
    StructField("unit", StringType()), StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()), StructField("genai_tokens", LongType()),  # v2 nullable
])
orgs_schema = StructType([
    StructField("org_id",StringType()),StructField("org_name",StringType()),
    StructField("industry",StringType()),StructField("hq_region",StringType()),
    StructField("plan_tier",StringType()),StructField("is_enterprise",BooleanType()),
    StructField("signup_date",DateType()),StructField("sales_rep",StringType()),
    StructField("lifecycle_stage",StringType()),StructField("marketing_source",StringType()),
    StructField("nps_score",DoubleType())])
users_schema = StructType([
    StructField("user_id",StringType()),StructField("org_id",StringType()),
    StructField("email",StringType()),StructField("role",StringType()),
    StructField("active",BooleanType()),StructField("created_at",DateType()),
    StructField("last_login",DateType())])
billing_schema = StructType([
    StructField("invoice_id",StringType()),StructField("org_id",StringType()),
    StructField("month",DateType()),StructField("subtotal",DoubleType()),
    StructField("credits",DoubleType()),StructField("taxes",DoubleType()),
    StructField("currency",StringType()),StructField("exchange_rate_to_usd",DoubleType())])
print("esquemas listos")

### 1b. Perfilado de datos (evidencia sobre landing crudo)
Corremos un perfilado rápido para **mostrar con números reales** los problemas de calidad que el pipeline ataca (esto también respalda que trabajamos sobre el dato real, no genérico).

In [ ]:
# Perfilado de eventos (lectura batch del landing, sólo para evidencia)
prof = (spark.read.schema(event_schema).json(EVENTS_DIR)
        .withColumn("value_num", F.col("value").cast("double")))
tot = prof.count()
print("eventos totales:", tot, "| event_id únicos:", prof.select("event_id").distinct().count())
prof.groupBy("schema_version").count().orderBy("schema_version").show()
print("value nulos:", prof.filter(F.col("value").isNull()).count())
print("unit nulo con value presente (viola regla 3):",
      prof.filter(F.col("value").isNotNull() & F.col("unit").isNull()).count())
print("cost_usd_increment negativos:", prof.filter(F.col("cost_usd_increment")<0).count())
prof.select(F.round(F.min("cost_usd_increment"),2).alias("min_cost"),
            F.round(F.max("cost_usd_increment"),2).alias("max_cost"),
            F.round(F.expr("percentile_approx(cost_usd_increment,0.99)"),2).alias("p99")).show()
# v1 NO trae la clave carbon_kg -> el esquema explícito la rellena como NULL (debería ser 0 no-nulos en v1)
print("filas v1 con carbon_kg no nulo (esperado 0):",
      prof.filter((F.col("schema_version")==1) & F.col("carbon_kg").isNotNull()).count())

# Perfilado de un par de maestros (issues que NO afectan el mart FinOps, sólo se documentan)
orgs_raw = spark.read.option("header",True).schema(orgs_schema).csv(f"{LANDING}/customers_orgs.csv")
bill_raw = spark.read.option("header",True).schema(billing_schema).csv(f"{LANDING}/billing_monthly.csv")
print("orgs NPS fuera de rango (>100):", orgs_raw.filter(F.col("nps_score")>100).count(),
      "| billing subtotales negativos:", bill_raw.filter(F.col("subtotal")<0).count(),
      "| billing credits vacíos:", bill_raw.filter(F.col("credits").isNull()).count())

# integridad referencial: eventos cuyo org_id no existe en orgs (quedarían con industry/plan nulos tras el join)
org_ids = orgs_raw.select("org_id").distinct()
sin_org = prof.select("org_id").distinct().join(org_ids, on="org_id", how="left_anti").count()
print("org_id de eventos que no existen en orgs (huérfanos):", sin_org)

## 2. Batch a Bronze
Bronze mantiene el mismo grano que el CSV pero con tipos explícitos, dedupe por la clave, y dos columnas de procedencia (`ingest_ts`, `source_file`). Lo guardamos en Parquet, que es columnar y va bien para lo analítico.

Particionamos cada maestro por una columna de baja cardinalidad (`hq_region`, `role`, `currency`) para cumplir el requisito de Parquet particionado. En este MVP nadie filtra los maestros por esas columnas, así que la elección es más por el requisito que por una consulta concreta; en producción se elegiría según los patrones de query.

In [ ]:
def csv_a_bronze(path, schema, key, out, part_col=None):
    df = (spark.read.option("header",True).schema(schema).csv(path)
          .withColumn("ingest_ts",F.current_timestamp())
          .withColumn("source_file",F.input_file_name())
          .dropDuplicates([key]))
    w = df.write.mode("overwrite")
    if part_col: w = w.partitionBy(part_col)     # Parquet particionado (requisito 1)
    w.parquet(out); print(f"{out}: {df.count()} filas (particionado por {part_col})"); return df

orgs    = csv_a_bronze(f"{LANDING}/customers_orgs.csv", orgs_schema,   "org_id",    f"{BRONZE}/orgs",    "hq_region")
users   = csv_a_bronze(f"{LANDING}/users.csv",          users_schema,  "user_id",   f"{BRONZE}/users",   "role")
billing = csv_a_bronze(f"{LANDING}/billing_monthly.csv",billing_schema,"invoice_id",f"{BRONZE}/billing", "currency")

## 3. Streaming a Bronze
Structured Streaming lee la carpeta de JSONLs como un stream: cada archivo nuevo es un micro-batch. Aplicamos `withWatermark` sobre `event_ts` (event time, no processing time) para acotar la late data, y dedupe por `event_id`, con checkpoint habilitado.

Usamos `trigger(availableNow=True)` porque la fuente es un set fijo de archivos: procesa todo lo que hay y termina, así la corrida es reproducible y no deja un stream infinito colgado en el notebook. (`readStream` no ejecuta nada por sí solo — la query arranca con `writeStream.start()`.)

In [ ]:
raw = spark.readStream.schema(event_schema).json(EVENTS_DIR)
bronze_stream = (raw
    .withColumn("event_ts",  F.to_timestamp("timestamp"))
    .withColumn("value_num", F.col("value").cast("double"))   # "100.0"/99.0 -> double; basura -> null
    .withColumn("usage_date",F.to_date("event_ts"))
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
    .withWatermark("event_ts","10 minutes")
    .dropDuplicates(["event_id"]))
q = (bronze_stream.writeStream.format("parquet")
     .option("path", f"{BRONZE}/events")
     .option("checkpointLocation", f"{CHK}/bronze_events")
     .outputMode("append").trigger(availableNow=True).start())
q.awaitTermination()
print("streaming -> bronze OK")

In [ ]:
bronze_events = spark.read.parquet(f"{BRONZE}/events")
print("eventos en bronze:", bronze_events.count())
bronze_events.select("event_id","event_ts","org_id","service","metric","value","value_num",
                     "cost_usd_increment","schema_version","genai_tokens").show(5, truncate=False)

## 4. Silver — calidad + quarantine + enriquecimiento
3 reglas de calidad; las filas que fallan van a **quarantine** (se inspeccionan, no se descartan). Además un `dropDuplicates(["event_id"])` por lotes como red global (la dedupe del stream sólo cubre la ventana del watermark).

Reglas: `event_id` no nulo · `cost_usd_increment ≥ -0.01` · `unit` no nulo cuando `value` existe.

Nota sobre la regla 3: manda a quarantine filas con `unit` nulo aunque tengan un costo válido, así que subestima un poco el costo en Gold. Medimos cuánto cuesta esa decisión abajo. La dejamos así porque la consigna pide esa regla, pero queda documentado el trade-off (en producción se podría conservar el costo y aislar sólo el `value`).

In [ ]:
ev = spark.read.parquet(f"{BRONZE}/events").dropDuplicates(["event_id"])
valida = (F.col("event_id").isNotNull()
          & (F.col("cost_usd_increment") >= -0.01)
          & ~(F.col("value").isNotNull() & F.col("unit").isNull()))   # regla 3
silver_ok  = ev.filter(valida)
quarantine = ev.filter(~valida)
quarantine.write.mode("overwrite").parquet(f"{QUARANTINE}/events")
print("válidas:", silver_ok.count(), "| en quarantine:", quarantine.count())
# trade-off de la regla 3: costo que se va a quarantine teniendo cost válido
costo_q = quarantine.filter(F.col("cost_usd_increment") >= 0) \
                    .agg(F.round(F.sum("cost_usd_increment"),2).alias("usd")).first()["usd"]
print("costo (USD) en filas quarantineadas con cost válido:", costo_q)
quarantine.select("event_id","value","unit","cost_usd_increment").show(5, truncate=False)

### Enriquecimiento (broadcast join)
Hacemos `broadcast` de la tabla de orgs (~80 filas) para evitar el shuffle en el join — Spark la manda a cada executor en vez de mover los eventos por la red.

In [ ]:
orgs_dim = spark.read.parquet(f"{BRONZE}/orgs").select(
    "org_id","industry","plan_tier","hq_region","lifecycle_stage")
silver = silver_ok.join(F.broadcast(orgs_dim), on="org_id", how="left")
print("silver enriquecido:", silver.count())

### 4b. Anomalías de costo — z-score / MAD / p99
Los tres métodos los pide la consigna. El z-score marca de más por la cola de outliers, p99 es medio bruto, MAD aguanta mejor; por eso marcamos sólo cuando coinciden **≥2 de 3** (baja el ruido de ~211 a ~89). Los estadísticos van **por servicio** porque los costos no están en la misma escala. Guardamos qué métodos dispararon en `anomaly_methods` — después lo usamos como `set<text>` en Cassandra. No recortamos el costo: marcamos y seguimos. (`1.4826` lleva el MAD a escala de desvío; `K=1.5` es el factor sobre p99 que elegimos nosotros.)

In [ ]:
stats = silver.groupBy("service").agg(
    F.mean("cost_usd_increment").alias("mu"), F.stddev("cost_usd_increment").alias("sd"),
    F.expr("percentile_approx(cost_usd_increment,0.5)").alias("med"),
    F.expr("percentile_approx(cost_usd_increment,0.99)").alias("p99"))
s1 = silver.join(F.broadcast(stats), on="service", how="left") \
           .withColumn("abs_dev", F.abs(F.col("cost_usd_increment")-F.col("med")))
mad = s1.groupBy("service").agg(F.expr("percentile_approx(abs_dev,0.5)").alias("mad"))
s2  = s1.join(F.broadcast(mad), on="service", how="left")

K = 1.5  # factor sobre p99
flag_z   = (F.abs(F.col("cost_usd_increment")-F.col("mu"))/F.col("sd")) > 3
flag_mad = (F.col("mad")>0) & ((F.col("abs_dev")/(1.4826*F.col("mad"))) > 3)
flag_p99 = F.col("cost_usd_increment") > (F.col("p99")*K)

# guardamos qué métodos dispararon -> después va como set<text> en Cassandra
silver_flagged = (s2
    .withColumn("anomaly_methods", F.array_compact(F.array(
        F.when(flag_z,   F.lit("zscore")),
        F.when(flag_mad, F.lit("mad")),
        F.when(flag_p99, F.lit("p99")))))
    .withColumn("anomaly", F.size("anomaly_methods") >= 2))

silver_flagged.write.mode("overwrite").partitionBy("usage_date","service").parquet(f"{SILVER}/events")
print("anomalías marcadas:", silver_flagged.filter("anomaly").count())
silver_flagged.filter("anomaly").select(
    "event_id","service","cost_usd_increment","anomaly_methods").orderBy(
    F.desc("cost_usd_increment")).show(10, truncate=False)

## 5. Gold — mart FinOps diario
Grano org×servicio×día. **Features derivadas de `metric`** (no un `count(*)` de filas). Redondeo aplicado en el borde de serving (los decimales largos de `sum()` sobre `double` son artefactos de punto flotante; el dinero se redondea a centavos).

In [ ]:
se = spark.read.parquet(f"{SILVER}/events")
gold_finops = (se.groupBy("org_id","service","usage_date").agg(
        F.round(F.sum("cost_usd_increment"),2).alias("daily_cost_usd"),
        F.round(F.sum(F.when(F.col("metric")=="requests", F.col("value_num"))),2).alias("requests"),
        F.round(F.sum(F.when(F.col("metric")=="cpu_hours", F.col("value_num"))),4).alias("cpu_hours"),
        F.round(F.sum(F.when(F.col("metric")=="storage_gb_hours", F.col("value_num"))),4).alias("storage_gb_hours"),
        F.sum("genai_tokens").alias("genai_tokens"),
        F.round(F.sum("carbon_kg"),6).alias("carbon_kg"),
        # set de métodos que dispararon ese día (vacío si no hubo anomalía) -> set<text> en Cassandra
        F.array_distinct(F.flatten(F.collect_set("anomaly_methods"))).alias("anomaly_methods"))
    .fillna(0, subset=["requests","cpu_hours","storage_gb_hours","genai_tokens","carbon_kg"]))
gold_finops.write.mode("overwrite").partitionBy("usage_date").parquet(f"{GOLD}/org_daily_usage_by_service")
print("filas en gold:", gold_finops.count())
gold_finops.orderBy(F.desc("daily_cost_usd")).show(8, truncate=False)

## 6. Serving — Cassandra/AstraDB (tabla CQL)
Modelado **query-first**: `PRIMARY KEY ((org_id, service), usage_date)` — la clave de partición `(org_id, service)` siempre va en el `WHERE`, y `usage_date` (clustering) permite el rango y el orden. La escritura es **UPSERT** (idempotente).

La tabla usa un tipo **colección** de CQL: `anomaly_methods set<text>` guarda qué métodos (zscore/mad/p99) marcaron anomalía ese día — un conjunto sin orden ni repetidos, que es justo lo que representa un `set`.

Credenciales: se leen de **Colab Secrets** o `.env` (nunca hardcodeadas). `dict_factory` indexa las filas como diccionarios (`r["col"]`).

In [ ]:
# Carga de credenciales: Colab Secrets -> .env -> os.environ (en ese orden de prioridad)
import os
try:
    from dotenv import load_dotenv; load_dotenv()  # lee un archivo .env si existe
except Exception:
    pass
def get_secret(clave, default=None):
    try:
        from google.colab import userdata
        v = userdata.get(clave)
        if v: return v
    except Exception:
        pass
    return os.getenv(clave, default)

SCB_PATH    = get_secret("SCB_PATH", "/content/secure-connect-YOURDB.zip")
ASTRA_TOKEN = get_secret("ASTRA_TOKEN")            # definir en Secrets/.env (AstraCS:...)
KEYSPACE    = get_secret("ASTRA_KEYSPACE", "cloud_analytics")
assert ASTRA_TOKEN, "Falta ASTRA_TOKEN: cargalo en Colab Secrets o en .env (ver .env.example)"
print("credenciales cargadas | keyspace:", KEYSPACE)

In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra.query import dict_factory
cluster = Cluster(cloud={"secure_connect_bundle": SCB_PATH},
                  auth_provider=PlainTextAuthProvider("token", ASTRA_TOKEN))
session = cluster.connect(KEYSPACE)
session.row_factory = dict_factory
print("conectado:", session.execute("select release_version from system.local").one())

In [ ]:
session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.org_daily_usage_by_service (
  org_id text, service text, usage_date date,
  daily_cost_usd double, requests double, cpu_hours double, storage_gb_hours double,
  genai_tokens bigint, carbon_kg double,
  anomaly_methods set<text>,
  PRIMARY KEY ((org_id, service), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)''')
print("tabla lista")

In [ ]:
import datetime
upsert = session.prepare(f'''
INSERT INTO {KEYSPACE}.org_daily_usage_by_service
(org_id, service, usage_date, daily_cost_usd, requests, cpu_hours, storage_gb_hours, genai_tokens, carbon_kg, anomaly_methods)
VALUES (?,?,?,?,?,?,?,?,?,?)''')

def cargar_finops():
    n=0
    for r in gold_finops.toLocalIterator():
        ud=r["usage_date"]
        if isinstance(ud, datetime.datetime): ud=ud.date()
        session.execute(upsert,(r["org_id"], r["service"], ud,
            float(r["daily_cost_usd"] or 0), float(r["requests"] or 0),
            float(r["cpu_hours"] or 0), float(r["storage_gb_hours"] or 0),
            int(r["genai_tokens"] or 0), float(r["carbon_kg"] or 0),
            set(r["anomaly_methods"] or [])))   # set<text> de métodos que dispararon
        n+=1
    return n
print("filas upserteadas:", cargar_finops())

## 7. Consultas Q1 y Q2
Los rangos de fechas se derivan **del propio dato** (sin año hardcodeado, para evitar el desfasaje 2025-vs-2026). Cada consulta imprime su CQL junto al resultado.

- **Q1**: costos y requests diarios por org+servicio en un rango (lectura de una sola partición).
- **Q2**: top-N servicios por costo acumulado en los últimos 14 días para una org. Como el top-N es sobre un valor calculado que cruza particiones, se resuelve leyendo las particiones de la org y agregando del lado de la app (la alternativa de producción es un mart dedicado `top_services_by_org`).

In [ ]:
import pandas as pd, datetime
muestra = gold_finops.orderBy(F.desc("daily_cost_usd")).first()
ORG, SVC = muestra["org_id"], muestra["service"]
limites = gold_finops.agg(F.min("usage_date").alias("lo"), F.max("usage_date").alias("hi")).first()
LO, HI = limites["lo"], limites["hi"]
print(f"Q1 muestra: org={ORG} service={SVC} rango={LO}..{HI}")

q1_cql = f'''SELECT usage_date, daily_cost_usd, requests
FROM {KEYSPACE}.org_daily_usage_by_service
WHERE org_id='{ORG}' AND service='{SVC}'
  AND usage_date>='{LO}' AND usage_date<='{HI}';'''
print("-- Q1 (CQL)\n"+q1_cql)

filas = session.execute(f'''SELECT usage_date, daily_cost_usd, requests
  FROM {KEYSPACE}.org_daily_usage_by_service
  WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s''', (ORG,SVC,LO,HI))
pd.DataFrame(list(filas))

In [ ]:
corte = HI - datetime.timedelta(days=14)
print(f"-- Q2 (CQL, repetida por servicio) ventana {corte}..{HI}")
servicios = sorted({r["service"] for r in session.execute(
    f"SELECT org_id, service FROM {KEYSPACE}.org_daily_usage_by_service") if r["org_id"]==ORG})
totales=[]
for s in servicios:
    tot = sum((r["daily_cost_usd"] or 0) for r in session.execute(
        f'''SELECT daily_cost_usd FROM {KEYSPACE}.org_daily_usage_by_service
            WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s''',
        (ORG, s, corte, HI)))
    totales.append((s, round(tot,4)))
pd.DataFrame(sorted(totales, key=lambda x:-x[1]), columns=["service","cost_14d"])

## 8. Idempotencia
Re-ejecutar la carga no duplica: los archivos son replayables, el stream tiene checkpoint, y la escritura a Cassandra es UPSERT por la clave `(org_id, service, usage_date)`. Mostramos el conteo antes y después — si es idempotente, no cambia.

In [ ]:
antes = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_by_service").one()["c"]
cargar_finops()
despues  = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_by_service").one()["c"]
print(f"antes={antes} despues={despues} -> idempotente: {antes==despues}")

## 8b. Evidencia de rutas y tamaños (particionado)
Requisito 6: mostramos las rutas de cada zona, su tamaño en disco y las carpetas de partición generadas (`hq_region=`, `usage_date=`, `service=`).

In [ ]:
import subprocess
def evidencia(z):
    du = subprocess.run(["du","-sh",z], capture_output=True, text=True).stdout.strip()
    parts = [p for p in subprocess.run(["ls",z], capture_output=True, text=True).stdout.split()
             if "=" in p]
    print(du)
    if parts: print("   particiones:", parts[:6], ("… (+%d)"%(len(parts)-6)) if len(parts)>6 else "")
for z in [f"{BRONZE}/orgs", f"{BRONZE}/users", f"{BRONZE}/billing", f"{BRONZE}/events",
          f"{SILVER}/events", f"{GOLD}/org_daily_usage_by_service"]:
    evidencia(z)

## 9. Speed Layer — streaming agregado → Cassandra con `foreachBatch`
Cassandra no es un sink nativo de Structured Streaming, así que usamos `foreachBatch` para mandar cada micro-batch con UPSERTs. Agregamos por ventana diaria (tumbling) sobre event-time.

Escribimos a una **tabla aparte** (`org_daily_usage_stream`), no al mart del batch: la capa de velocidad sólo calcula costo y requests, así que si escribiera a la misma tabla pondría en cero `cpu_hours`/`storage`/`genai` y pisaría los datos completos que cargó el batch. `outputMode("update")` porque la agregación se actualiza con cada batch.

In [ ]:
# La Speed Layer escribe a SU PROPIA tabla, para no pisar las columnas del mart batch.
session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.org_daily_usage_stream (
  org_id text, service text, usage_date date,
  daily_cost_usd double, requests double,
  PRIMARY KEY ((org_id, service), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)''')
upsert_stream = session.prepare(f'''
INSERT INTO {KEYSPACE}.org_daily_usage_stream
(org_id, service, usage_date, daily_cost_usd, requests) VALUES (?,?,?,?,?)''')

stream_silver = (spark.readStream.schema(event_schema).json(EVENTS_DIR)
    .withColumn("event_ts", F.to_timestamp("timestamp"))
    .withColumn("value_num", F.col("value").cast("double"))
    .filter((F.col("event_id").isNotNull()) & (F.col("cost_usd_increment") >= -0.01))
    .withWatermark("event_ts", "10 minutes"))

stream_gold = (stream_silver
    .groupBy(F.window("event_ts", "1 day"), "org_id", "service")   # ventana tumbling diaria
    .agg(F.sum("cost_usd_increment").alias("daily_cost_usd"),
         F.sum(F.when(F.col("metric")=="requests", F.col("value_num"))).alias("requests"))
    .withColumn("usage_date", F.to_date(F.col("window.start"))))

def a_cassandra(batch_df, batch_id):
    for r in batch_df.collect():
        session.execute(upsert_stream, (r["org_id"], r["service"], r["usage_date"],
            float(r["daily_cost_usd"] or 0), float(r["requests"] or 0)))

qs = (stream_gold.writeStream
      .outputMode("update")
      .foreachBatch(a_cassandra)
      .option("checkpointLocation", f"{CHK}/speed_finops")
      .trigger(availableNow=True)
      .start())
qs.awaitTermination()
n_stream = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_stream").one()["c"]
print("speed layer OK · filas en org_daily_usage_stream:", n_stream)

## 10. Reporte de evidencia
Esta celda consolida todos los criterios.

In [ ]:
# ================================================================
#  REPORTE DE EVIDENCIA — MVP (Segundo Parcial)
#  Cloud Provider Analytics · Camila Lee (63382) · Lucas Perri (62746)
#  Correr al final, con AstraDB conectado y las zonas generadas.
# ================================================================
import pandas as pd, datetime, subprocess
def _sec(n, txt):
    print("\n" + "="*66); print(f"  CRITERIO {n} · {txt}"); print("="*66)

# --- 1. Batch y streaming corren con datos provistos ---
_sec(1, "Batch y streaming corren con datos provistos")
for z in ["orgs","users","billing"]:
    print(f"  Bronze {z:<8} : {spark.read.parquet(f'{BRONZE}/{z}').count():>6} filas  (batch)")
print(f"  Bronze events  : {spark.read.parquet(f'{BRONZE}/events').count():>6} filas  (streaming)")

# --- 2. Reglas de calidad y quarantine efectivas ---
_sec(2, "Reglas de calidad y quarantine efectivas (ejemplos)")
ev = spark.read.parquet(f"{BRONZE}/events").dropDuplicates(["event_id"])
val = (F.col("event_id").isNotNull() & (F.col("cost_usd_increment")>=-0.01)
       & ~(F.col("value").isNotNull() & F.col("unit").isNull()))
print(f"  Válidas    : {ev.filter(val).count():>6}")
print(f"  Quarantine : {ev.filter(~val).count():>6}  (filas que violan ≥1 regla, aisladas)")
ev.filter(~val).select("event_id","value","unit","cost_usd_increment").show(5, truncate=False)

# --- 3. Mart FinOps en Gold + tabla en Cassandra poblada ---
_sec(3, "Mart FinOps en Gold + tabla en Cassandra poblada")
g = spark.read.parquet(f"{GOLD}/org_daily_usage_by_service")
n_cas = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_by_service").one()["c"]
print(f"  Filas en Gold (mart FinOps)      : {g.count():>6}")
print(f"  Filas en Cassandra (org_daily…)  : {n_cas:>6}")
g.orderBy(F.desc("daily_cost_usd")).show(5, truncate=False)

# parámetros derivados del dato (no del reloj)
top = g.orderBy(F.desc("daily_cost_usd")).first(); ORG, SVC = top["org_id"], top["service"]
bb = g.agg(F.min("usage_date").alias("lo"), F.max("usage_date").alias("hi")).first()
LO, HI = bb["lo"], bb["hi"]

# --- 4. Q1: CQL + resultado ---
_sec(4, "Consulta Q1 sobre AstraDB (CQL + resultado)")
print("-- Q1 (CQL)")
print(f"SELECT usage_date, daily_cost_usd, requests")
print(f"FROM {KEYSPACE}.org_daily_usage_by_service")
print(f"WHERE org_id='{ORG}' AND service='{SVC}'")
print(f"  AND usage_date>='{LO}' AND usage_date<='{HI}';\n")
r1 = session.execute(f"SELECT usage_date,daily_cost_usd,requests FROM {KEYSPACE}.org_daily_usage_by_service "
                     f"WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s",(ORG,SVC,LO,HI))
print(pd.DataFrame(list(r1)).to_string(index=False))

# --- 5. Q2: CQL + resultado ---
_sec(5, "Consulta Q2 sobre AstraDB (CQL + resultado)")
corte = HI - datetime.timedelta(days=14)
print(f"-- Q2 (CQL, por servicio de la org · ventana 14 días · org={ORG})")
print(f"SELECT daily_cost_usd FROM {KEYSPACE}.org_daily_usage_by_service")
print(f"WHERE org_id='{ORG}' AND service=? AND usage_date>='{corte}' AND usage_date<='{HI}';\n")
servs = sorted({r["service"] for r in session.execute(
    f"SELECT org_id,service FROM {KEYSPACE}.org_daily_usage_by_service") if r["org_id"]==ORG})
tot=[]
for s in servs:
    t=sum((r["daily_cost_usd"] or 0) for r in session.execute(
        f"SELECT daily_cost_usd FROM {KEYSPACE}.org_daily_usage_by_service "
        f"WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s",(ORG,s,corte,HI)))
    tot.append((s,round(t,2)))
print(pd.DataFrame(sorted(tot,key=lambda x:-x[1]),columns=["service","cost_14d"]).to_string(index=False))

# --- 6. Idempotencia ---
_sec(6, "Reprocesar no duplica (idempotencia)")
antes = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_by_service").one()["c"]
try: cargar_finops()
except Exception as e: print("  (re-carga omitida:", e, ")")
despues = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_by_service").one()["c"]
print(f"  Conteo antes   : {antes}")
print(f"  Conteo después : {despues}")
print(f"  Idempotente    : {antes==despues}")

# --- (+) Evidencia de rutas y tamaños ---
_sec("+", "Rutas y tamaños del particionado")
for z in [f"{BRONZE}/events", f"{SILVER}/events", f"{GOLD}/org_daily_usage_by_service"]:
    du = subprocess.run(["du","-sh",z],capture_output=True,text=True).stdout.strip()
    parts=[p for p in subprocess.run(["ls",z],capture_output=True,text=True).stdout.split() if "=" in p]
    print(f"  {du}")
    if parts: print(f"     particiones: {parts[:4]}{' …' if len(parts)>4 else ''}")
print("\n" + "="*66 + "\n  FIN DEL REPORTE DE EVIDENCIA\n" + "="*66)

## 11. Observabilidad y cierre
Miramos el `lastProgress` de la query de ingesta (cuántas filas entraron, estado del watermark) y cerramos los streams con `stop()`. Dejar streams infinitos colgados en el notebook es problema.

In [ ]:
print("lastProgress (ingesta):", q.lastProgress)
for s in spark.streams.active: s.stop()
print("streams activos:", spark.streams.active)